In [2]:
# import important libaries
import pandas as pd 
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder , LabelEncoder
from sklearn.compose import ColumnTransformer

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD

In [20]:
# read data from local file
df = pd.read_csv(r"C:\Users\Lenovo\Downloads\covid_toy.csv")
df 

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No
...,...,...,...,...,...,...
95,12,Female,104.0,Mild,Bangalore,No
96,51,Female,101.0,Strong,Kolkata,Yes
97,20,Female,101.0,Mild,Bangalore,No
98,5,Female,98.0,Strong,Mumbai,No


In [5]:
df["fever"].fillna(df["fever"].mean(),inplace = True)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4488\3056235766.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["fever"].fillna(df["fever"].mean(),inplace = True)


In [6]:
# make a target columns
X = df.drop("has_covid", axis=1)
y = df["has_covid"]

In [7]:
# Split into Training and Testing
cat = X.select_dtypes(include=["object"]).columns
num = X.select_dtypes(exclude=["object"]).columns

In [8]:
# pretranform data 
pre = ColumnTransformer(
    transformers=[
        ("cat" , OneHotEncoder(drop= "first" , handle_unknown="ignore" ), cat),
        ("num", StandardScaler(),num)
    ]
)
pre

,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,'first'
,sparse_output,True


In [9]:
# Split into Training and Testing
from sklearn.model_selection import train_test_split
X_train , X_test , y_train , y_test = train_test_split(X,y ,test_size=0.2, random_state=42)

In [10]:
# Fitting data into preprocess
X_train = pre.fit_transform(X_train)
X_test = pre.transform(X_test)

In [23]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [24]:
model = Sequential()
# Input Layer
# model.add(Input(shape=(X_train.shape[1],)))

# Hidden Layer 1
model.add(Dense(32, input_dim =X_train.shape[1], activation='relu'))

# Hidden Layer 2
model.add(Dense(16, activation='relu'))

# Output Layer
model.add(Dense(1, activation='sigmoid'))

C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [25]:
sgd = SGD(
    learning_rate=0.01,
    momentum=0.9
)

model.compile(
    optimizer=sgd,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [26]:
y_train.dtype

dtype('int64')

In [27]:
model.fit(X_train , y_train , epochs=30 , batch_size=20 , verbose=1 , validation_data= (X_test , y_test), validation_split=0.2)

Epoch 1/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 77ms/step - accuracy: 0.5250 - loss: 0.7022 - val_accuracy: 0.4000 - val_loss: 0.7069
Epoch 2/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5375 - loss: 0.6994 - val_accuracy: 0.4500 - val_loss: 0.7029
Epoch 3/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5750 - loss: 0.6958 - val_accuracy: 0.4500 - val_loss: 0.6964
Epoch 4/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5875 - loss: 0.6919 - val_accuracy: 0.4500 - val_loss: 0.6901
Epoch 5/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.6000 - loss: 0.6887 - val_accuracy: 0.5500 - val_loss: 0.6830
Epoch 6/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.6250 - loss: 0.6843 - val_accuracy: 0.6500 - val_loss: 0.6787
Epoch 7/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6375 - loss: 0.6809 - val_accuracy: 0.7000 - val_loss: 0.6749
Epoch 8/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.6125 - loss: 0.6778 - val_accuracy: 0.7000 - val_loss: 0.6710


In [29]:
# check accuarcy of model
loss, accuracy = model.evaluate(X_test, y_test)

print(f"Loss: {loss}")
print(f"Accuracy: {accuracy * 100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.6500 - loss: 0.6517
Loss: 0.651728630065918
Accuracy: 65.00%
